In [1]:
# Import helpers
import sys
import os
sys.path.append(os.path.join(os.path.dirname('.'), '..'))

# Preprocessing
from preprocess_pipeline import (
    MNEPipeline, notch, bandpass, rereference,
    add_idle_class, add_fake_jaw_clench, epoch, extract_tensor,
)
import numpy as np
import json
import itertools
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.model_selection import StratifiedKFold

# Importing models
from ml_model.models import (
    DeepConvNet, ShallowConvNet, EEGNet, ATCNet, Conformer, CTNet, MBMANet, PBT, LMDANet, EEGITNet
)

import torch
import torch.nn as nn

In [2]:
# =============================================================================
# Configurations
# =============================================================================

N_CHANNELS   = 8    # Number of EEG channels in your data
N_CLASSES    = 3    # idle, move, jaw_clench
SFREQ        = 300  # Sample rate
TRIAL_DUR    = 3.0  # seconds (tmax - tmin in epoch())
N_TIMEPOINTS = int(SFREQ * TRIAL_DUR)

FILES = [
    '../data_collection/annotated_eeg/chengyi0210_eeg.fif',
    '../data_collection/annotated_eeg/pilapil0226_eeg.fif',
]
LABEL_MAP = {'idle': 0, 'move': 1, 'jaw_clench': 2}
MAP_LABEL = {v: k for k, v in LABEL_MAP.items()}

BANDPASS_CONFIGS = [
    [(8, 13)],
    [(13, 30)],
    [(8, 13), (13, 30)],
    [(4, 8), (8, 13), (13, 30)],
]

EPOCHS     = 100
BATCH_SIZE = 32
LR         = 1e-3
CV_FOLDS   = 5
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {DEVICE}  |  N_CHANNELS: {N_CHANNELS}  |  N_TIMEPOINTS: {N_TIMEPOINTS}")

Device: cuda  |  N_CHANNELS: 8  |  N_TIMEPOINTS: 900


In [3]:
# =============================================================================
# Load data
# =============================================================================

def load_tensor_data(files, bands, label_map):
    all_X, all_y = [], []
    for file in files:
        pipeline = MNEPipeline(file)
        pipeline.add(notch(60))
        pipeline.add(bandpass(bands))
        pipeline.add(rereference())
        pipeline.add(add_idle_class(window_dur=3.0, idle_start_min=1.0))
        pipeline.add(add_fake_jaw_clench())
        pipeline.add(epoch(tmin=0, tmax=TRIAL_DUR))

        raw = pipeline.run()
        X, y = extract_tensor(scale=True)(raw)   # (N, 1, C, T)

        auto_ids = raw._event_id
        remap = {v: label_map[k] for k, v in auto_ids.items() if k in label_map}
        y = np.array([remap[yi] for yi in y])

        T = X.shape[-1]
        if T > N_TIMEPOINTS:
            X = X[..., :N_TIMEPOINTS]
        elif T < N_TIMEPOINTS:
            pad = np.zeros((*X.shape[:-1], N_TIMEPOINTS - T), dtype=np.float32)
            X = np.concatenate([X, pad], axis=-1)

        all_X.append(X)
        all_y.append(y)

    X = np.concatenate(all_X, axis=0)
    y = np.concatenate(all_y, axis=0)
    print(f"  Loaded: X={X.shape}, classes={np.unique(y, return_counts=True)}")
    return X, y

X, y = load_tensor_data(FILES, BANDPASS_CONFIGS[0], LABEL_MAP)

Opening raw data file ../data_collection/annotated_eeg/chengyi0210_eeg.fif...
Isotrak not found
    Range : 0 ... 433799 =      0.000 ...  1445.997 secs
Ready.
Reading 0 ... 433799  =      0.000 ...  1445.997 secs...


Applying: <function notch.<locals>.apply at 0x71584926a8e0>
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 1981 samples (6.603 s)

Applying: <function bandpass.<locals>.apply at 0x71583c5a6200>
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 13 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuatio

/home/chengyi/Projects/Weimo/processing/preprocess_pipeline.py:307: RuntimeWarning: Invalid tag with only 0/16 bytes at position 1205359 in file /home/chengyi/Projects/Weimo/processing/../data_collection/annotated_eeg/pilapil0226_eeg.fif
  raw = mne.io.read_raw_fif(self.fif_file, preload=True)


In [4]:
X.shape, y.shape # (N, 1, Channels, Seconds*SFREQ), (N,)

((336, 1, 8, 900), (336,))

In [5]:
# Show class distribution
unique, counts = np.unique(y, return_counts=True)
print("Class distribution:")
for cls, count in zip(unique, counts):
    print(f"  Class {MAP_LABEL[cls]}: {count} samples")    

Class distribution:
  Class idle: 237 samples
  Class move: 97 samples
  Class jaw_clench: 2 samples


In [6]:
# DeepConvNet(n_channels, n_classes)

# ShallowConvNet(n_channels, n_classes)

# EEGNet(n_channels, n_classes, n_timepoints, sampling_rate, num_temporal_filters=8, num_spatial_filters=10)

# ATCNet(n_channels, n_classes, sampling_rate, num_filters=16, d=2, p2=8, num_heads=2, block_one_dropout=0.3, attn_drouput=0.5)

# Conformer(n_channels, n_classes, n_timepoints, emb_size=40, depth=6)

# CTNet(n_channels, n_classes, n_timepoints, sampling_rate, num_temporal_filters=8, D=2, dropout=0.5, d=16)

# MBMANet(n_channels, n_classes, n_timepoints, sampling_rate, F1s=(4,8,16), D=2, tcn_channels=64, tcn_layers=2, dropout=0.5)

# PBT(d_input, n_classes, num_embeddings, num_tokens_per_channel, d_model, n_blocks, num_heads, dropout, device, learnable_cls=False, bias_transformer=False, bert=False)

# LMDANet(n_channels, n_classes, n_timepoints, depth=9, kernel=25, channel_depth1=24, channel_depth2=9, ave_depth=1, avepool=5, dropout=0.65)

# EEGITNet(n_channels, n_classes, n_timepoints, n_inception=2, out_per_branch=8, D=2, tcn_channels=16, tcn_layers=2, dropout=0.5)

In [7]:
deepconv_model   = DeepConvNet(N_CHANNELS, N_CLASSES).to(DEVICE)
shallowconv_model = ShallowConvNet(N_CHANNELS, N_CLASSES).to(DEVICE)
eegnet_model     = EEGNet(N_CHANNELS, N_CLASSES, N_TIMEPOINTS, SFREQ).to(DEVICE)
atcnet_model     = ATCNet(N_CHANNELS, N_CLASSES, SFREQ).to(DEVICE)
conformer_model  = Conformer(N_CHANNELS, N_CLASSES, N_TIMEPOINTS).to(DEVICE)
ctnet_model      = CTNet(N_CHANNELS, N_CLASSES, N_TIMEPOINTS, SFREQ).to(DEVICE)
mbmanet_model    = MBMANet(N_CHANNELS, N_CLASSES, N_TIMEPOINTS, SFREQ).to(DEVICE)
lmda_model       = LMDANet(N_CHANNELS, N_CLASSES, N_TIMEPOINTS).to(DEVICE)
itnet_model      = EEGITNet(N_CHANNELS, N_CLASSES, N_TIMEPOINTS).to(DEVICE)

/home/chengyi/anaconda3/envs/VLM/lib/python3.12/site-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv2d(


In [8]:
# =============================================================================
# Training utilities
# =============================================================================

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
        correct   += (out.argmax(1) == yb).sum().item()
        n         += len(yb)
    return total_loss / n, correct / n


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss = criterion(out, yb)
        total_loss += loss.item() * len(yb)
        correct   += (out.argmax(1) == yb).sum().item()
        n         += len(yb)
    return total_loss / n, correct / n


def run_cv(model_fn, X, y, config):
    """
    model_fn : callable with no args → returns a fresh model
    config   : dict with keys epochs, batch_size, lr, cv_folds, device, patience
    """
    device    = config['device']
    criterion = nn.CrossEntropyLoss()
    skf       = StratifiedKFold(n_splits=config['cv_folds'], shuffle=True, random_state=42)

    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)
    dataset = TensorDataset(Xt, yt)

    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n── Fold {fold+1}/{config['cv_folds']} ──")

        train_loader = DataLoader(Subset(dataset, train_idx),
                                  batch_size=config['batch_size'], shuffle=True)
        val_loader   = DataLoader(Subset(dataset, val_idx),
                                  batch_size=config['batch_size'])

        model     = model_fn().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                        optimizer, T_max=config['epochs'])

        best_val_loss  = float('inf')
        best_state     = None
        patience_count = 0

        for epoch in range(1, config['epochs'] + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
            va_loss, va_acc = eval_epoch(model, val_loader, criterion, device)
            scheduler.step()

            if epoch % 10 == 0:
                print(f"  ep {epoch:3d} | tr {tr_loss:.4f}/{tr_acc:.3f} "
                      f"| va {va_loss:.4f}/{va_acc:.3f}")

            # Early stopping
            if va_loss < best_val_loss - 1e-4:
                best_val_loss  = va_loss
                best_state     = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_count = 0
            else:
                patience_count += 1
                if patience_count >= config['patience']:
                    print(f"  Early stop at epoch {epoch}")
                    break

        model.load_state_dict(best_state)
        _, best_val_acc = eval_epoch(model, val_loader, criterion, device)
        fold_results.append({'val_acc': best_val_acc, 'val_loss': best_val_loss, 'model': model})
        print(f"  Best val acc: {best_val_acc:.4f}")

    accs = [r['val_acc'] for r in fold_results]
    print(f"\nCV result: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    return fold_results


# =============================================================================
# Model registry
# =============================================================================

MODEL_REGISTRY = {
    'DeepConvNet'  : lambda: DeepConvNet(N_CHANNELS, N_CLASSES),
    'ShallowConvNet': lambda: ShallowConvNet(N_CHANNELS, N_CLASSES),
    'EEGNet'       : lambda: EEGNet(N_CHANNELS, N_CLASSES, N_TIMEPOINTS, SFREQ),
    'ATCNet'       : lambda: ATCNet(N_CHANNELS, N_CLASSES, SFREQ),
    'Conformer'    : lambda: Conformer(N_CHANNELS, N_CLASSES, N_TIMEPOINTS),
    'CTNet'        : lambda: CTNet(N_CHANNELS, N_CLASSES, N_TIMEPOINTS, SFREQ),
    'MBMANet'      : lambda: MBMANet(N_CHANNELS, N_CLASSES, N_TIMEPOINTS, SFREQ),
    'LMDANet'      : lambda: LMDANet(N_CHANNELS, N_CLASSES, N_TIMEPOINTS),
    'EEGITNet'     : lambda: EEGITNet(N_CHANNELS, N_CLASSES, N_TIMEPOINTS),
}

CONFIG = {
    'epochs'     : EPOCHS,
    'batch_size' : BATCH_SIZE,
    'lr'         : LR,
    'cv_folds'   : CV_FOLDS,
    'device'     : DEVICE,
    'patience'   : 15,          # stop if val loss doesn't improve for 15 epochs
}

# =============================================================================
# Run all models
# =============================================================================

all_results = {}

for name, model_fn in MODEL_REGISTRY.items():
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    fold_results = run_cv(model_fn, X, y, CONFIG)
    all_results[name] = fold_results

# =============================================================================
# Summary table
# =============================================================================

print(f"\n{'Model':<16} {'Mean Acc':>10} {'Std':>8}")
print('-' * 36)
for name, results in all_results.items():
    accs = [r['val_acc'] for r in results]
    print(f"{name:<16} {np.mean(accs):>10.4f} {np.std(accs):>8.4f}")

/home/chengyi/anaconda3/envs/VLM/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(



DeepConvNet

── Fold 1/5 ──
  ep  10 | tr 0.6324/0.925 | va 0.7314/0.809
  ep  20 | tr 0.5820/0.981 | va 0.6854/0.853
  ep  30 | tr 0.5588/0.993 | va 0.6913/0.853
  ep  40 | tr 0.5561/0.996 | va 0.6758/0.882
  ep  50 | tr 0.5523/1.000 | va 0.6801/0.882
  ep  60 | tr 0.5515/1.000 | va 0.6748/0.868
  Early stop at epoch 68
  Best val acc: 0.9118

── Fold 2/5 ──
  ep  10 | tr 0.6228/0.944 | va 0.7849/0.761
  ep  20 | tr 0.5712/0.981 | va 0.7276/0.836
  ep  30 | tr 0.5872/0.970 | va 0.8174/0.731
  Early stop at epoch 35
  Best val acc: 0.8358

── Fold 3/5 ──
  ep  10 | tr 0.6340/0.926 | va 0.7514/0.821
  ep  20 | tr 0.5719/0.985 | va 0.7579/0.791
  ep  30 | tr 0.5601/0.993 | va 0.7368/0.836
  ep  40 | tr 0.5576/0.996 | va 0.7380/0.836
  ep  50 | tr 0.5563/0.996 | va 0.7302/0.836
  ep  60 | tr 0.5571/0.996 | va 0.7551/0.791
  Early stop at epoch 62
  Best val acc: 0.8358

── Fold 4/5 ──
  ep  10 | tr 0.6225/0.952 | va 1.0018/0.552
  ep  20 | tr 0.5673/0.985 | va 0.7146/0.836
  ep  30 | tr 

/home/chengyi/anaconda3/envs/VLM/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


  ep  10 | tr 0.8077/0.746 | va 0.8434/0.691
  ep  20 | tr 0.7688/0.806 | va 0.8507/0.676
  Early stop at epoch 28
  Best val acc: 0.6765

── Fold 2/5 ──
  ep  10 | tr 0.8101/0.747 | va 0.8240/0.746
  ep  20 | tr 0.7667/0.825 | va 0.8174/0.731
  ep  30 | tr 0.7435/0.848 | va 0.8121/0.746
  ep  40 | tr 0.7362/0.855 | va 0.8045/0.731
  ep  50 | tr 0.7202/0.885 | va 0.8064/0.746
  ep  60 | tr 0.7097/0.885 | va 0.8001/0.746
  ep  70 | tr 0.6958/0.896 | va 0.7956/0.761
  Early stop at epoch 78
  Best val acc: 0.7612

── Fold 3/5 ──
  ep  10 | tr 0.8077/0.758 | va 0.8543/0.687
  ep  20 | tr 0.7711/0.803 | va 0.8595/0.627
  ep  30 | tr 0.7511/0.829 | va 0.8582/0.642
  ep  40 | tr 0.7311/0.851 | va 0.8462/0.657
  ep  50 | tr 0.7137/0.888 | va 0.8426/0.687
  ep  60 | tr 0.7041/0.896 | va 0.8423/0.672
  ep  70 | tr 0.6929/0.903 | va 0.8354/0.716
  ep  80 | tr 0.6878/0.926 | va 0.8351/0.716
  Early stop at epoch 88
  Best val acc: 0.7313

── Fold 4/5 ──
  ep  10 | tr 0.8188/0.717 | va 0.8187/0.71

/home/chengyi/anaconda3/envs/VLM/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


...

  ep  10 | tr 0.6986/0.698 | va 0.5544/0.750
  ep  20 | tr 0.4569/0.754 | va 0.6313/0.706
  Early stop at epoch 25
  Best val acc: 0.7500

── Fold 2/5 ──
  ep  10 | tr 0.6038/0.755 | va 0.8119/0.672
  Early stop at epoch 18
  Best val acc: 0.6269

── Fold 3/5 ──
  ep  10 | tr 0.6749/0.710 | va 0.8539/0.627
  Early stop at epoch 18
  Best val acc: 0.6866

── Fold 4/5 ──
  ep  10 | tr 0.6003/0.706 | va 0.5784/0.731
  ep  20 | tr 0.4649/0.784 | va 0.6050/0.701
  ep  30 | tr 0.4761/0.792 | va 0.7627/0.731
  Early stop at epoch 31
  Best val acc: 0.7463

── Fold 5/5 ──
  ep  10 | tr 0.6459/0.714 | va 0.6578/0.672
  ep  20 | tr 0.4295/0.840 | va 0.8215/0.672
  Early stop at epoch 25
  Best val acc: 0.6716

CV result: 0.6963 ± 0.0467

LMDANet

── Fold 1/5 ──


/home/chengyi/anaconda3/envs/VLM/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


  ep  10 | tr 0.3918/0.840 | va 0.4751/0.794
  ep  20 | tr 0.2297/0.922 | va 0.4366/0.868
  ep  30 | tr 0.1737/0.933 | va 0.4634/0.882
  Early stop at epoch 31
  Best val acc: 0.8676

── Fold 2/5 ──
  ep  10 | tr 0.4195/0.818 | va 0.5975/0.746
  ep  20 | tr 0.2365/0.892 | va 0.5754/0.746
  ep  30 | tr 0.1851/0.922 | va 0.5922/0.746
  ep  40 | tr 0.1630/0.941 | va 0.5783/0.746
  Early stop at epoch 40
  Best val acc: 0.7313

── Fold 3/5 ──
  ep  10 | tr 0.4471/0.777 | va 0.6644/0.716
  ep  20 | tr 0.2670/0.896 | va 0.5900/0.731
  ep  30 | tr 0.1774/0.944 | va 0.5974/0.746
  Early stop at epoch 36
  Best val acc: 0.7164

── Fold 4/5 ──
  ep  10 | tr 0.4113/0.807 | va 0.4788/0.776
  ep  20 | tr 0.2522/0.903 | va 0.4303/0.806
  ep  30 | tr 0.1665/0.948 | va 0.4043/0.836
  ep  40 | tr 0.1050/0.955 | va 0.3790/0.851
  ep  50 | tr 0.1376/0.941 | va 0.3877/0.836
  ep  60 | tr 0.0740/0.978 | va 0.3940/0.866
  Early stop at epoch 66
  Best val acc: 0.8806

── Fold 5/5 ──
  ep  10 | tr 0.4268/0.8

/home/chengyi/anaconda3/envs/VLM/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


  ep  10 | tr 0.7972/0.746 | va 0.8455/0.662
  ep  20 | tr 0.7310/0.817 | va 0.8003/0.779
  ep  30 | tr 0.6962/0.858 | va 0.7783/0.794
  ep  40 | tr 0.6708/0.888 | va 0.7580/0.779
  ep  50 | tr 0.6314/0.914 | va 0.7744/0.765
  Early stop at epoch 50
  Best val acc: 0.7941

── Fold 2/5 ──
  ep  10 | tr 0.7848/0.777 | va 0.8789/0.657
  ep  20 | tr 0.7039/0.844 | va 0.7663/0.761
  ep  30 | tr 0.6664/0.885 | va 0.7849/0.746
  ep  40 | tr 0.6409/0.907 | va 0.7560/0.776
  ep  50 | tr 0.6306/0.922 | va 0.7363/0.821
  Early stop at epoch 53
  Best val acc: 0.8209

── Fold 3/5 ──
  ep  10 | tr 0.7926/0.777 | va 0.8365/0.701
  ep  20 | tr 0.7136/0.840 | va 0.7965/0.761
  ep  30 | tr 0.6549/0.914 | va 0.7910/0.761
  Early stop at epoch 31
  Best val acc: 0.7612

── Fold 4/5 ──
  ep  10 | tr 0.7982/0.766 | va 0.8611/0.716
  ep  20 | tr 0.7274/0.825 | va 0.8281/0.716
  ep  30 | tr 0.7031/0.851 | va 0.7888/0.761
  ep  40 | tr 0.6850/0.866 | va 0.8042/0.746
  Early stop at epoch 46
  Best val acc: 0.